# **G2P Evaluation of a ByT5 Model on Wiktionary Phonemic Data**

This notebook tests a neural G2P model,
[`lowestofthelow/filipino-byt5-g2p`](https://huggingface.co/lowestofthelow/filipino-byt5-g2p), on
the same held-out test split as `Wik_eval.ipynb`. That makes its scores directly comparable with
the WFST model behind the `tagalog_g2p.py` command-line tool. It is the existing-model neural
candidate in `G2P_compare.ipynb`; `ByT5_finetune.ipynb` is the controlled one, a neural model
trained on the same data as the WFST.

In this notebook, "ByT5" means this model, `filipino-byt5-g2p`. The fine-tuned CharsiuG2P of
`ByT5_finetune.ipynb` is also a ByT5 model, so `G2P_compare.ipynb` uses the full names.

The model comes from Marqueses et al. (2026), *Towards Stress-Aware Sentence-Level Filipino G2P
With Weakly-Supervised ByT5 Fine-Tuning* ([arXiv:2609.09974](https://arxiv.org/abs/2609.09974),
NLPIR 2026), with code and data at
[dlsu-cse-nlp/filipino-byt5-g2p](https://github.com/dlsu-cse-nlp/filipino-byt5-g2p). It is a ByT5
transformer that reads text one byte at a time and writes IPA the same way. It starts from
CharsiuG2P (`charsiu/g2p_multilingual_byT5_small_100`; Zhu et al., 2022), a word-level G2P model
for about 100 languages, and is fine-tuned for **sentence-level**, stress-aware Filipino G2P. Its
training sentences (Tatoeba, NewsPH-NLI and LLM-generated sentences) were transcribed by an
LLM-assisted pipeline that looks words up in a WikiPron scrape of Wiktionary's Tagalog entries. The
paper reports a PER of 0.54% on its own manually corrected sentence test set.

This has two consequences here:

- The model is built for sentences, but the test split has single words. Each word is given to it
  on its own, in the authors' input format (section 3). The stress marks it adds are removed
  before scoring, since the references are scored without stress.
- Its training labels come from the same Wiktionary data as the test split, so it has seen many
  of the test words. Section 8 measures this and scores seen and unseen words separately.

The checkpoint is the Hub commit
[`ebd4663a44766d0a97a23e79bcca5ba91bcaebfc`](https://huggingface.co/lowestofthelow/filipino-byt5-g2p/tree/ebd4663a44766d0a97a23e79bcca5ba91bcaebfc)
of 20 August 2026, the latest release when this notebook was run. The paper doesn't say which of its
trained configurations this is. `MODEL_REVISION` pins it, so a rerun loads the same weights even if
the repository is updated.

Run `Wik_eval.ipynb` first: it writes the test split (`dicts/cwik_test.tsv`), the validation split
(`dicts/cwik_val.dict`), the cross-validation pool (`dicts/cwik_trainval.dict`) and the WFST predictions this notebook compares against. Unlike
the WFST notebooks, this one doesn't need Phonetisaurus, but it does need `torch` and
`transformers` (`pip install -r requirements.txt`), which the Docker image doesn't include. The
model (1.2 GB) is downloaded once into the Hugging Face cache, and section 8 downloads the authors'
training data (about 10 MB) from GitHub. A GPU is used if there is one, but a CPU works too.

**The process at a glance**

1. **Load** the test split written by `Wik_eval.ipynb`.
2. **Load** the model from Hugging Face.
3. **Choose** the input format, comparing the authors' format with a few others on part of the
   validation split.
4. **Predict** every test word and save the predictions to `output/byt5_output.txt`.
5. **Score** the predictions with the same PER and WER code as `Wik_eval.ipynb`.
6. **Score** each of the 10 cross-validation folds of `Wik_eval.ipynb`, to check how stable the
   PER is.
7. **Compare** the result with the WFST model.
8. **Check** how many test words the model saw in training, and score them separately.
9. **Analyze** the errors, with the same breakdowns as `Wik_eval.ipynb`.

In [1]:
import os
import random
import re
import sys
import time
import unicodedata
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer, T5ForConditionalGeneration
from transformers.utils import logging as hf_logging

sys.dont_write_bytecode = True  # no __pycache__ in ../utils
sys.path.append(os.path.abspath("../utils"))
from tsv_metrics import tokenize_ipa, compute_per_wer, collect_errors, vowel_confusion
from alignment import align_trxn

hf_logging.set_verbosity_error()  # hide checkpoint warnings and progress bars
hf_logging.disable_progress_bar()

current_dir = Path().resolve()
dicts_dir   = current_dir / "dicts"
output_dir  = current_dir / "output"

/workspaces/Taglog-G2P/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Load the test split

`Wik_eval.ipynb` saves its test split to `dicts/cwik_test.tsv`, one word per line with its phones
separated by spaces. The reference (`ref`) is built exactly as that notebook builds it, so both
models are scored against the same words and transcriptions.

The split has **2,171** words, each listed once.

In [2]:
test_tsv = dicts_dir / "cwik_test.tsv"
if not test_tsv.is_file():
    raise FileNotFoundError(f"{test_tsv} not found. Run Wik_eval.ipynb first: it writes the test split.")

# Same reference Wik_eval scores against: the phones are already space-separated, and a word
# listed twice keeps its last transcription.
ref = {}
n_entries = 0
with open(test_tsv, encoding="utf-8") as f:
    for line in f:
        word, phones = line.rstrip("\n").split("\t")
        ref[word] = phones.split()
        n_entries += 1

print(f"Test entries     : {n_entries}")
print(f"Unique test words: {len(ref)}")

Test entries     : 2171
Unique test words: 2171


## 2. Load the model

The model and its tokenizer are downloaded from Hugging Face. `predict()` sends the words through
in batches of 64 and decodes greedily (one beam); section 3 checks this against the five beams
of the authors' inference script.

The model's output needs three fixes before it can be compared with the references:

- It marks stress with an apostrophe (`ka'misa`), the paper's stress marker, which
  `tokenize_ipa()` doesn't strip, so `to_phones()` removes it.
- It writes sentences, so its output can contain spaces. A single word should come out as one
  word, but spaces are removed too, so any extra words count as errors.
- It sometimes writes an affricate without the tie bar (`baɾdʒen`), which `tokenize_ipa()` would
  split into two phones. The references always use the tie bar, so `dʒ` and `tʃ` become `d͡ʒ` and
  `t͡ʃ`. Otherwise the model uses the same symbols as the references, including `ɾ` for the tap
  and `ɡ` (not the letter `g`).

`tokenize_ipa()` then splits the string into phones, the same way the references were made.

In [3]:
MODEL_ID = "lowestofthelow/filipino-byt5-g2p"
MODEL_REVISION = "ebd4663a44766d0a97a23e79bcca5ba91bcaebfc"   # Hub commit of 2026-08-20

device = "cuda" if torch.cuda.is_available() else "cpu"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=MODEL_REVISION)
model = T5ForConditionalGeneration.from_pretrained(MODEL_ID, revision=MODEL_REVISION)
model = model.to(device).eval()

print("Device:", device)
print(f"Parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.0f}M")


def capitalize(text):
    """The authors' input format: the text as is, with its first letter capitalized."""
    return text[:1].upper() + text[1:]


FORMAT = capitalize   # chosen in section 3


def predict(words, fmt=FORMAT, num_beams=1, batch_size=64):
    """Return {word: raw model output} for the words formatted with fmt."""
    raw = {}
    for i in range(0, len(words), batch_size):
        batch = words[i:i + batch_size]
        enc = tokenizer([fmt(w) for w in batch], padding=True, return_tensors="pt").to(device)
        with torch.no_grad():
            out = model.generate(**enc, num_beams=num_beams, max_new_tokens=128)
        raw.update(zip(batch, tokenizer.batch_decode(out, skip_special_tokens=True)))
    return raw


def to_phones(ipa):
    """Tokenize model output like the Wik_eval references: no spaces, no ' stress marks, and
    affricates with the tie bar."""
    ipa = "".join(ipa.split()).replace("'", "")
    return tokenize_ipa(ipa.replace("dʒ", "d͡ʒ").replace("tʃ", "t͡ʃ"))


def read_output(path):
    """Read a word TAB phones file into {word: [phones]}."""
    preds = {}
    with open(path, encoding="utf-8") as f:
        for line in f:
            parts = line.strip().split("\t")
            if len(parts) >= 2:
                preds[parts[0]] = parts[1].split()
    return preds

Device: cpu
Parameters: 301M


## 3. Choose the input format

The authors train and run the model on plain text with no language tag: each sentence keeps its
spelling, and only its first letter is capitalized (`capitalize_sentence()` in their
`src/utils/dataset_from_csv.py`). For a single word that means a capitalized word, like `Kamisa`.
Their inference script also decodes with 5 beams. A few options are scored on the first 300 words
of the validation split (`dicts/cwik_val.dict`), so the test split isn't used to choose anything:

| Input | What it tests |
| --- | --- |
| `Kamisa` | the authors' format |
| `Kamisa`, 5 beams | the authors' format with their beam search |
| `Kamisa.` | the word as a one-word sentence, with a full stop |
| `kamisa` | the word in lowercase, as it appears in the test split |
| `<fil>: kamisa` | CharsiuG2P's Filipino language tag, the base model's format |

The three capitalized inputs are all close: `Kamisa` gives a PER of **1.67%**, and the authors'
5 beams or a full stop lower it to 1.63%, about one phone in the 300 words. So the authors' own
format, `Kamisa` with greedy decoding, is kept: it is the documented format, and greedy decoding is
much faster on a CPU. The base model's `<fil>: ` tag is worse (2.00%), and a lowercase word fails
outright: the model adds extra words to 65 of the 300 outputs and the PER rises to 75.17%, likely
because its training sentences always start with a capital letter.

In [4]:
val_ref = {}
with open(dicts_dir / "cwik_val.dict", encoding="utf-8") as f:
    for line in f:
        word, phones = line.rstrip("\n").split("\t")
        val_ref[word] = phones.split()

probe_words = list(val_ref)[:300]
probe_ref = {w: val_ref[w] for w in probe_words}

options = [
    ("Kamisa",        capitalize,                     1),
    ("Kamisa",        capitalize,                     5),
    ("Kamisa.",       lambda w: capitalize(w) + ".",  1),
    ("kamisa",        lambda w: w,                    1),
    ("<fil>: kamisa", lambda w: "<fil>: " + w,        1),
]

rows = []
for name, fmt, beams in options:
    raw = predict(probe_words, fmt=fmt, num_beams=beams)
    m = compute_per_wer(probe_ref, {w: to_phones(s) for w, s in raw.items()})
    rows.append({
        "Input": name,
        "Beams": beams,
        "PER (%)": round(m["per"] * 100, 2),
        "WER (%)": round(m["wer"] * 100, 2),
        "Outputs with spaces": sum(" " in s.strip() for s in raw.values()),
        "Example (kamisa)": predict(["kamisa"], fmt=fmt, num_beams=beams)["kamisa"],
    })

pd.DataFrame(rows)

,Input,Beams,PER (%),WER (%),Outputs with spaces,Example (kamisa)
0,Kamisa,1,1.67,10.33,0,ka'misa
1,Kamisa,5,1.63,10.00,0,ka'misa
2,Kamisa.,1,1.63,9.67,0,ka'misa
3,kamisa,1,75.17,44.00,65,ka'misa
4,<fil>: kamisa,1,2.00,12.00,1,ka'misa


## 4. Predict the test words

Every test word goes through the model with the chosen format. The predictions are saved to
`output/byt5_output.txt` in the same `word TAB phones` format as the WFST predictions.

All 2,171 words are predicted in a few minutes on a CPU. The examples show the fixes from
section 2 at work: the stress mark in `ka'misa` is dropped. `kato` loses its word-final `ʔ`, the
same ambiguity the WFST model faces, and `alkaide` gets the vowel `i` where the reference has the
glide `j`.

In [5]:
start = time.time()
raw = predict(list(ref))
hyp = {w: to_phones(s) for w, s in raw.items()}
print(f"Predicted {len(hyp)} words in {time.time() - start:.0f} s")

output_dir.mkdir(exist_ok=True)
byt5_output = output_dir / "byt5_output.txt"
with open(byt5_output, "w", encoding="utf-8") as f:
    for word, phones in hyp.items():
        f.write(f"{word}\t{' '.join(phones)}\n")

for word in [w for w in ["kato", "kamisa", "alkaide", "kutsada"] if w in ref]:
    print(f"  {word:<10} raw:[{raw[word]}]  phones:[{' '.join(hyp[word])}]  ref:[{' '.join(ref[word])}]")

Predicted 2171 words in 138 s
  kato       raw:[ka'to]  phones:[k a t o]  ref:[k a t o ʔ]
  kamisa     raw:[ka'misa]  phones:[k a m i s a]  ref:[k a m i s a]
  alkaide    raw:[ʔalkai'de]  phones:[ʔ a l k a i d e]  ref:[ʔ a l k a j d e]


## 5. Score the predictions

PER and WER are computed with `compute_per_wer()`, the function `Wik_eval.ipynb` uses. An `assert`
checks that every test word got a prediction.

ByT5 reaches a PER of **1.07%** and a WER of **6.77%**: 2,024 of the 2,171 test words are fully
correct. Every test word got a prediction.

In [6]:
metrics = compute_per_wer(ref, hyp)

# Every test word should get a prediction.
assert metrics["n_words"] == len(ref), f"scored {metrics['n_words']} of {len(ref)} test words"

print("=== ByT5 (filipino-byt5-g2p) on the Wik_eval held-out test split ===")
print(f"  PER : {metrics['per']:.4f}  ({metrics['per'] * 100:.2f}%)   <- primary metric")
print(f"  WER : {metrics['wer']:.4f}  ({metrics['wer'] * 100:.2f}%)")
print(f"  Words evaluated  : {metrics['n_words']}")
print(f"  Correct words    : {metrics['n_correct']}")
print(f"  Total ref phones : {metrics['n_ref_phones']}")
print(f"  Total edit dist  : {metrics['n_edit']}")

=== ByT5 (filipino-byt5-g2p) on the Wik_eval held-out test split ===
  PER : 0.0107  (1.07%)   <- primary metric
  WER : 0.0677  (6.77%)
  Words evaluated  : 2171
  Correct words    : 2024
  Total ref phones : 17303
  Total edit dist  : 186


## 6. Per-fold evaluation

The same 10 folds as the cross-validation in section 5 of `Wik_eval.ipynb`: the 90% training and
validation pool from `dicts/cwik_trainval.dict`, shuffled with the same seed and cut into the same
folds. There, the WFST model is retrained on nine folds and tested on the tenth. ByT5 is already
trained and can't be retrained the same way, so each fold is only scored, without retraining. This
isn't a cross-validation, but the spread across the folds still shows how stable its PER is on
different sets of words.

This predicts all the words in the pool, about nine times the test split, so it takes a while on a
CPU. The predictions are saved to `output/byt5_cv_output.txt` and reused on the next run (delete
the file to predict again), and the fold PERs to `output/byt5_cv.tsv` for `G2P_compare.ipynb`.

The mean PER across the folds is **1.21%** (± 0.13%), against 0.99% (± 0.06%) for the WFST model.
ByT5's PER is higher than the WFST's on every fold, and varies about twice as much from fold to
fold.

In [7]:
trainval_dict = dicts_dir / "cwik_trainval.dict"
if not trainval_dict.is_file():
    raise FileNotFoundError(f"{trainval_dict} not found. Run Wik_eval.ipynb first: it writes the pool.")

# Rebuild Wik_eval's folds: the pool in file order, shuffled with the same seed.
trainval = []
with open(trainval_dict, encoding="utf-8") as f:
    for line in f:
        word, phones = line.rstrip("\n").split("\t")
        trainval.append((word, phones.split()))

random.seed(42)
cv_entries = trainval[:]
random.shuffle(cv_entries)

N_FOLDS = 10
fold_size = len(cv_entries) // N_FOLDS
folds = [cv_entries[i * fold_size:(i + 1) * fold_size] for i in range(N_FOLDS)]
folds[-1] += cv_entries[N_FOLDS * fold_size:]   # remainder into last fold
print("Fold sizes:", [len(f) for f in folds])

# ByT5 isn't retrained per fold, so each pool word is predicted once and shared by the folds.
cv_output = output_dir / "byt5_cv_output.txt"
cv_hyp = read_output(cv_output) if cv_output.is_file() else {}
todo = list(dict.fromkeys(w for w, _ in cv_entries if w not in cv_hyp))
if todo:
    start = time.time()
    cv_hyp.update({w: to_phones(s) for w, s in predict(todo).items()})
    print(f"Predicted {len(todo)} words in {time.time() - start:.0f} s")
    with open(cv_output, "w", encoding="utf-8") as f:
        for word, phones in cv_hyp.items():
            f.write(f"{word}\t{' '.join(phones)}\n")
else:
    print(f"Reused the predictions in {cv_output.name}")

fold_pers = []
for k, held_out in enumerate(folds):
    fold_ref = {w: p for w, p in held_out}
    m = compute_per_wer(fold_ref, cv_hyp)
    assert m["n_words"] == len(fold_ref), f"fold {k + 1}: scored {m['n_words']} of {len(fold_ref)} words"
    fold_pers.append(m["per"])
    print(f"  Fold {k + 1:2d}: PER {m['per'] * 100:5.2f}%  ({m['n_words']} words)")

fold_pers = np.array(fold_pers)

# Saved for G2P_compare.ipynb.
pd.DataFrame({"fold": range(1, N_FOLDS + 1), "per": fold_pers}).to_csv(
    output_dir / "byt5_cv.tsv", sep="\t", index=False)
print()
print(f"Per-fold PER: mean {fold_pers.mean() * 100:.2f}%  +/-  {fold_pers.std() * 100:.2f}%")

Fold sizes: [1953, 1953, 1953, 1953, 1953, 1953, 1953, 1953, 1953, 1962]


Predicted 1704 words in 122 s
  Fold  1: PER  1.00%  (1953 words)


  Fold  2: PER  1.11%  (1953 words)
  Fold  3: PER  1.38%  (1953 words)


  Fold  4: PER  1.27%  (1953 words)
  Fold  5: PER  1.08%  (1953 words)


  Fold  6: PER  1.16%  (1953 words)
  Fold  7: PER  1.43%  (1953 words)


  Fold  8: PER  1.17%  (1953 words)
  Fold  9: PER  1.23%  (1953 words)


  Fold 10: PER  1.21%  (1962 words)

Per-fold PER: mean 1.21%  +/-  0.13%


## 7. Compare with the WFST model

The WFST predictions in `output/cwik_output.txt` are scored against the same reference. That file
comes from `Wik_eval.ipynb`, so the table only has the WFST row after that notebook has run.

The WFST model is better on both metrics: a PER of **0.81%** against **1.07%**, and a WER of
**5.85%** against **6.77%**. ByT5 makes **186** phone edits to the WFST's 141 and gets 20 fewer
words fully correct. Section 8 shows that this overall gap hides a large difference between the
words ByT5 saw in training and the ones it didn't. `G2P_compare.ipynb` tests whether the gap is
significant.

In [8]:
rows = [{"Model": "ByT5 (filipino-byt5-g2p)", **metrics}]

cwik_output = output_dir / "cwik_output.txt"
if cwik_output.is_file():
    wfst = read_output(cwik_output)
    rows.insert(0, {"Model": "WFST (Wik_eval)", **compute_per_wer(ref, wfst)})
else:
    print(f"{cwik_output.name} not found: run Wik_eval.ipynb to add the WFST model to this table.")

summary = pd.DataFrame(rows)
summary["PER (%)"] = (summary["per"] * 100).round(2)
summary["WER (%)"] = (summary["wer"] * 100).round(2)
summary[["Model", "PER (%)", "WER (%)", "n_words", "n_correct", "n_ref_phones", "n_edit"]]

,Model,PER (%),WER (%),n_words,n_correct,n_ref_phones,n_edit
0,WFST (Wik_eval),0.81,5.85,2171,2044,17303,141
1,ByT5 (filipino-byt5-g2p),1.07,6.77,2171,2024,17303,186


## 8. Words seen in training

The model's training labels come from the same source as the test split: Wiktionary's Tagalog
entries, through WikiPron. Words the model has already seen are easier for it, so this section
measures the overlap with the authors' published data, pinned to commit `3e19bfc` of their
repository:

| Data | What it is |
| --- | --- |
| `wikipron/wikipron_tl.tsv` | their WikiPron word list, which the labelling pipeline looks words up in |
| Tatoeba, NewsPH-NLI and naive synthetic training splits | the published training sentences (the paper's fourth set, the Wiktionary-guided synthetic sentences, isn't published) |
| `multitask/multitask_ds.csv` | single words with their WikiPron pronunciations. The paper doesn't list it as training data, so it's unknown whether the released checkpoint used it |

A test word counts as **seen** if it appears in any of the published training sentences. Both
models are then scored on the seen and unseen words separately. The WFST model saw none of the
test words, so the gap between its two scores shows how much the groups differ in difficulty.

The overlap is large: **1,929** of the 2,171 test words (88.9%) are in the authors' WikiPron word
list, and **835** (38.5%) appear in the published training sentences. `multitask_ds.csv` covers
1,648 (75.9%).

ByT5 does better on the words it has seen: a PER of **0.91%** on the seen words against **1.17%**
on the unseen ones. The seen words aren't easier in themselves, since the WFST model does worse on
them (1.00% against 0.71%). So on seen words ByT5 beats the WFST model, but on unseen words the
WFST model is clearly better (**0.71%** against **1.17%**). The unseen words
are the fairer comparison. Even that score may be optimistic: the Wiktionary-guided sentences
aren't published, and the released checkpoint may have been trained on `multitask_ds.csv`, so
some "unseen" words may have been seen after all.

In [9]:
# The authors' published data, pinned to a commit of github.com/dlsu-cse-nlp/filipino-byt5-g2p.
FBYT5_DATA = ("https://media.githubusercontent.com/media/dlsu-cse-nlp/filipino-byt5-g2p/"
              "3e19bfc94246773e617c429318bb6d10bde6f8cb/data/")
TRAIN_SENTENCES = [
    "tatoeba/phonetic_tatoeba_gemini_3_train.csv",
    "newsph-nli/phonetic_newsph-nli_gemini_2.5_lite_train.csv",
    "stress-minimal/stress-minimal_ambiguous_split_train.csv",   # naive synthetic
    "stress-minimal/stress-minimal_single_split_train.csv",      # naive synthetic
]


def words_in(texts):
    """The lowercased words in a list of texts."""
    return {w for t in texts for w in re.findall(r"[\w'-]+", unicodedata.normalize("NFC", str(t)).lower())}


sentence_words = set()
for path in TRAIN_SENTENCES:
    sentence_words |= words_in(pd.read_csv(FBYT5_DATA + path)["sentence"])
wikipron_words = words_in(pd.read_csv(FBYT5_DATA + "wikipron/wikipron_tl.tsv", sep="\t",
                                      header=None, quoting=3)[0])
multitask_words = words_in(pd.read_csv(FBYT5_DATA + "multitask/multitask_ds.csv")["sentence"])

for label, vocab in [("their WikiPron word list", wikipron_words),
                     ("the published training sentences", sentence_words),
                     ("multitask_ds.csv", multitask_words)]:
    n = sum(w in vocab for w in ref)
    print(f"Test words in {label:<33}: {n:5d} of {len(ref)} ({n / len(ref) * 100:.1f}%)")

seen = [w for w in ref if w in sentence_words]
unseen = [w for w in ref if w not in sentence_words]

models = {"ByT5": hyp}
if cwik_output.is_file():
    models = {"WFST": wfst, **models}

rows = []
for name, preds in models.items():
    for group, words in [("seen", seen), ("unseen", unseen), ("all", list(ref))]:
        m = compute_per_wer({w: ref[w] for w in words}, preds)
        rows.append({"Model": name, "Test words": group, "Words": len(words),
                     "PER (%)": round(m["per"] * 100, 2), "WER (%)": round(m["wer"] * 100, 2)})
pd.DataFrame(rows)

Test words in their WikiPron word list         :  1929 of 2171 (88.9%)
Test words in the published training sentences :   835 of 2171 (38.5%)
Test words in multitask_ds.csv                 :  1648 of 2171 (75.9%)


,Model,Test words,Words,PER (%),WER (%)
0,WFST,seen,835,1.00,6.59
1,WFST,unseen,1336,0.71,5.39
2,WFST,all,2171,0.81,5.85
3,ByT5,seen,835,0.91,5.51
4,ByT5,unseen,1336,1.17,7.56
5,ByT5,all,2171,1.07,6.77


## 9. Error analysis

The same breakdowns as in `Wik_eval.ipynb`: by error type, by phone and by substitution pair,
then the glottal stop and vowel-height checks.

### 9.1 Error types

There are **186** edits in total: 32 substitutions, 75 deletions and 79 insertions. Compared
with the WFST model (13, 56 and 72), ByT5 makes more than twice as many substitutions and
somewhat more deletions.

In [10]:
err = collect_errors(ref, hyp)
total_errors = err["n_sub"] + err["n_del"] + err["n_ins"]

pd.DataFrame([
    {"Type": "Substitution", "Count": err["n_sub"], "% of Errors": f"{err['n_sub'] / total_errors * 100:.1f}%"},
    {"Type": "Deletion",     "Count": err["n_del"], "% of Errors": f"{err['n_del'] / total_errors * 100:.1f}%"},
    {"Type": "Insertion",    "Count": err["n_ins"], "% of Errors": f"{err['n_ins'] / total_errors * 100:.1f}%"},
    {"Type": "TOTAL",        "Count": total_errors, "% of Errors": "100.0%"},
])

,Type,Count,% of Errors
0,Substitution,32,17.2%
1,Deletion,75,40.3%
2,Insertion,79,42.5%
3,TOTAL,186,100.0%


### 9.2 Phones behind each error type

The glottal stop `ʔ` is again the phone behind most deletions and insertions: it is dropped
**62** times and added **44** times. Most substitutions are of `j` (11) and `t͡ʃ` (6).

In [11]:
def top10(counter):
    return pd.DataFrame([{"Phone": p, "Count": c} for p, c in counter.most_common(10)])

print("Ref phones most often substituted:")
display(top10(err["sub_ref_counter"]))
print("Ref phones most often deleted:")
display(top10(err["del_counter"]))
print("Phones most often inserted:")
display(top10(err["ins_counter"]))

Ref phones most often substituted:


,Phone,Count
0,j,11
1,t͡ʃ,6
2,e,3
3,i,2
4,ʔ,1
5,s,1
6,d,1
7,w,1
8,u,1
9,n,1


Ref phones most often deleted:


,Phone,Count
0,ʔ,62
1,s,5
2,u,1
3,h,1
4,w,1
5,ɡ,1
6,ŋ,1
7,ɾ,1
8,j,1
9,e,1


Phones most often inserted:


,Phone,Count
0,ʔ,44
1,t,8
2,j,4
3,w,4
4,e,3
5,u,2
6,p,2
7,o,2
8,s,2
9,i,2


### 9.3 Substitution pairs

The most common pair is `t͡ʃ → s` (6), where the model reads the spelling *ts* as /ts/ and the
reference has the affricate /t͡ʃ/ (`butsi`). Next are `j → i` (5) and `j → ʃ` (4), which come from a
different transcription convention rather than a wrong sound: `j → i` writes a glide as a full
vowel (`alkaide` as /ʔalkaide/), and `j → ʃ` turns the *sy* in `syokoy` into /ʃ/.

In [12]:
display(pd.DataFrame([
    {"Pair": f"{r} -> {h}", "Count": c}
    for (r, h), c in err["sub_pair_counter"].most_common(10)
]))

print("Example words for the top substitution pairs:")
for (r, h), c in err["sub_pair_counter"].most_common(5):
    print(f"\n  {r} -> {h}  (count={c})")
    for ex in err["sub_pair_examples"].get((r, h), [])[:3]:
        print(f"    {ex['word']:<20}  ref:[{ex['ref']}]  hyp:[{ex['hyp']}]")

,Pair,Count
0,t͡ʃ -> s,6
1,j -> i,5
2,j -> ʃ,4
3,e -> a,2
4,i -> j,2
5,ʔ -> w,1
6,s -> ʃ,1
7,d -> t,1
8,w -> ʔ,1
9,j -> e,1


Example words for the top substitution pairs:

  t͡ʃ -> s  (count=6)
    maghitso              ref:[m a ɡ h i t͡ʃ o]  hyp:[m a ɡ h i t s o]
    remolatsa             ref:[ɾ e m o l a t͡ʃ a]  hyp:[ɾ e m o l a t s a]
    butsi                 ref:[b u t͡ʃ i]  hyp:[b u t s i]

  j -> i  (count=5)
    proteina              ref:[p ɾ o t e j n a]  hyp:[p ɾ o t e ʔ i n a]
    alkaide               ref:[ʔ a l k a j d e]  hyp:[ʔ a l k a i d e]
    ispeir                ref:[ʔ i s p e j ɾ]  hyp:[ʔ i s p e ʔ i ɾ]

  j -> ʃ  (count=4)
    iniksyon              ref:[ʔ i n i k s j o n]  hyp:[ʔ i n i k ʃ o n]
    isine-share           ref:[ʔ i s i n e s j e j ɾ]  hyp:[ʔ i s i n e ʃ e ɾ]
    syokoy                ref:[s j o k o j]  hyp:[ʃ o k o j]

  e -> a  (count=2)
    deins                 ref:[d e j n s]  hyp:[d a j n s]
    geim                  ref:[ɡ e j m]  hyp:[ɡ a j m]

  i -> j  (count=2)
    tiraniya              ref:[t i ɾ a n i a]  hyp:[t i ɾ a n j a]
    tite                  ref:[t i 

### 9.4 Phones the references never use

The model was trained on a different transcription convention, so it can produce phones that the
Wiktionary references never use. Each one counts as an error even if it is a reasonable sound.

There are **none**: with the tie-bar fix from section 2, every predicted phone also appears in the
references. So the convention differences in section 9.3 swap one reference phone for another,
rather than bring in new symbols.

In [13]:
ref_inventory = {p for phones in ref.values() for p in phones}
hyp_counts = Counter(p for phones in hyp.values() for p in phones)
outside = {p: c for p, c in hyp_counts.most_common() if p not in ref_inventory}

print(f"Phones in the references : {len(ref_inventory)}")
print(f"Phones in the predictions: {len(hyp_counts)}")
print(f"Predicted phones never used in the references: {outside if outside else 'none'}")

for p in list(outside)[:5]:
    examples = [w for w, phones in hyp.items() if p in phones][:3]
    print(f"\n  {p}:")
    for w in examples:
        print(f"    {w:<20}  ref:[{' '.join(ref[w])}]  hyp:[{' '.join(hyp[w])}]")

Phones in the references : 26
Phones in the predictions: 25
Predicted phones never used in the references: none


### 9.5 Where the glottal stop goes wrong

The glottal stop `ʔ` is the main weakness of the WFST model, because Tagalog spelling doesn't mark
it at the end of a word. This counts each `ʔ` deletion and insertion by its position in the word,
for both models.

ByT5 drops `ʔ` more often than the WFST model (**62** against 50) but adds it less often (**44**
against 65). Its deletions are word-final as often as the WFST's or more (**93.5%** against
88.0%), but only **65.9%** of its insertions are, against 87.7%: it also adds `ʔ` inside words (14
times), more often than the WFST model (8).

In [14]:
GLOTTAL = "ʔ"


def glottal_positions(ref, hyp):
    """Count ʔ deletions (by ref position) and insertions (by hyp position): initial/medial/final."""
    def bucket(idx, n):
        return "initial" if idx == 0 else "final" if idx == n - 1 else "medial"

    del_pos, ins_pos = Counter(), Counter()
    for word, ref_phones in ref.items():
        hyp_phones = hyp.get(word, [])
        aln_ref, aln_hyp, _, _ = align_trxn(ref_phones, hyp_phones)
        ri = hi = 0
        for a, b in zip(aln_ref, aln_hyp):
            if a == GLOTTAL and b == "*":
                del_pos[bucket(ri, len(ref_phones))] += 1
            if b == GLOTTAL and a == "*":
                ins_pos[bucket(hi, len(hyp_phones))] += 1
            ri += a != "*"
            hi += b != "*"
    return del_pos, ins_pos


models = {"ByT5": hyp}
if cwik_output.is_file():
    models = {"WFST": wfst, **models}

rows = []
for name, preds in models.items():
    del_pos, ins_pos = glottal_positions(ref, preds)
    for kind, counter in [("deleted", del_pos), ("inserted", ins_pos)]:
        rows.append({"Model": name, "ʔ": kind, **{p: counter[p] for p in ("initial", "medial", "final")},
                     "Total": sum(counter.values())})
pd.DataFrame(rows)

,Model,ʔ,initial,medial,final,Total
0,WFST,deleted,1,5,44,50
1,WFST,inserted,0,8,57,65
2,ByT5,deleted,1,3,58,62
3,ByT5,inserted,1,14,29,44


### 9.6 Vowel-height confusion: o↔u and e↔i

Tagalog *o*/*u* and *e*/*i* were historically allophones, so a model can swap them with no cue
from the spelling. `vowel_confusion()` counts every o↔u and e↔i substitution, in both directions.

There is just **one**, an e→i swap: `reorganisasyon` starts with /ɾi/. There are no o↔u swaps, so
like the WFST model, ByT5 keeps these vowels apart.

In [15]:
vc = vowel_confusion(ref, hyp)   # default groups: o/u and e/i

display(pd.DataFrame([
    {"Confusion": k, "Count": v["count"], "% of subs": f"{v['rate'] * 100:.1f}%"}
    for k, v in vc.items()
]))

for label, v in vc.items():
    print(f"\n{label}: {v['count']} confusions")
    for (r, h), c in v["by_pair"].most_common():
        print(f"    {r} -> {h}: {c}")
    for ex in v["examples"][:5]:
        print(f"      {ex['word']:<18} ref:[{ex['ref']}]  hyp:[{ex['hyp']}]")

,Confusion,Count,% of subs
0,o/u,0,0.0%
1,e/i,1,3.1%



o/u: 0 confusions

e/i: 1 confusions
    e -> i: 1
      reorganisasyon     ref:[ɾ e ʔ o ɾ ɡ a n i s a s j o n]  hyp:[ɾ i ʔ o ɾ ɡ a n i s a s j o n]
